# Module 4: The Overfitting Trap
## Part 1: Running a Parameter Sweep

---

In this module, you're going to experience something that tricks **most** crypto quant researchers.

You'll run a parameter sweep, find amazing-looking results, get excited... and then discover it was all an illusion.

This is the most important module in the course. If you understand this, you'll be ahead of 95% of people building trading systems.

---

### What is a parameter sweep?

A parameter sweep tests many different combinations of entry rules to find the "best" one.

For example, if your strategy is:
- Enter long when `liquidation_z_score > X`
- AND `crowding_ratio < Y`
- AND `oi_divergence > Z`

A sweep tries every combination of X, Y, and Z to find which one makes the most money.

Sounds smart, right? **That's the trap.**

### Setup

First, let's load a real dataset. We'll use ETH 4h data with CoinGlass features.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import product

# Load the sample dataset (included in this repo)
DATA_DIR = Path('data')
sample_file = DATA_DIR / 'sample_eth_4h.parquet'

if sample_file.exists():
    df = pd.read_parquet(sample_file)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp').reset_index(drop=True)
    print(f'Loaded {len(df)} rows of ETH 4h data')
    print(f'Date range: {df["timestamp"].min()} to {df["timestamp"].max()}')
else:
    print(f'File not found: {sample_file}')
    print('Make sure the data/ folder is in the same directory as this notebook.')

### Step 1: Define the strategy template

Our strategy is a **liquidation-driven long** — when liquidations spike and the crowd isn't too long, we buy expecting a mean reversion bounce.

The parameters we'll sweep:

In [ ]:
# Parameter grid — every combination will be tested
param_grid = {
    'liq_z_threshold':    [0.3, 0.6, 1.0, 1.5, 2.0],    # How big the liquidation spike must be
    'liq_imbalance_min':  [0.0, 0.05, 0.10, 0.20],       # Minimum short-side liquidation dominance
    'crowding_cap':       [0.0, 0.2, 0.5, 0.7, 1.0],     # Maximum crowding (lower = stricter)
    'oi_divergence_min':  [-0.10, -0.05, -0.02, 0.0],     # Minimum OI-price divergence
}

# Count total combinations
total = 1
for values in param_grid.values():
    total *= len(values)

print(f'Parameters: {list(param_grid.keys())}')
print(f'Total combinations to test: {total}')
print()
print('This is the core of the problem.')
print(f'We are about to test {total} different strategies on the SAME data.')
print('At least one of them will look amazing just by random chance.')

### Step 2: Build walk-forward splits

We use walk-forward validation — train on the past, test on the future, slide forward.

This seems rigorous. It IS rigorous... for a single strategy. But when you sweep 400 combinations across the same folds, something sneaky happens.

In [ ]:
def build_walk_forward_splits(n_rows, train_bars=24*45, test_bars=24*14, step_bars=24*14):
    """Create temporal train/test splits."""
    splits = []
    start = 0
    while start + train_bars + test_bars <= n_rows:
        splits.append({
            'train_start': start,
            'train_end': start + train_bars,
            'test_start': start + train_bars,
            'test_end': start + train_bars + test_bars,
        })
        start += step_bars
    return splits

splits = build_walk_forward_splits(len(df))
print(f'Walk-forward folds: {len(splits)}')
for i, s in enumerate(splits):
    print(f'  Fold {i+1}: train [{s["train_start"]}:{s["train_end"]}] '
          f'test [{s["test_start"]}:{s["test_end"]}]')

### Step 3: Run the sweep

For each parameter combination, we test on the walk-forward folds and measure the average return.

This takes a minute. Watch the progress — we're testing hundreds of strategies.

In [ ]:
def evaluate_strategy(df, splits, liq_z, liq_imb, crowd_cap, oi_div, 
                      target='fwd_ret_24', hold_bars=24, cost_bps=12):
    """Evaluate one parameter combination across all walk-forward folds."""
    cost = cost_bps / 10000
    all_returns = []
    
    # Build entry mask
    mask = (
        (df.get('liq_total_z_72', pd.Series(dtype=float)) > liq_z) &
        (df.get('liquidation_imbalance', pd.Series(dtype=float)) > liq_imb) &
        (df.get('crowding_top_pos_z_72', pd.Series(dtype=float)) < crowd_cap) &
        (df.get('oi_price_divergence_24', pd.Series(dtype=float)) > oi_div)
    ).fillna(False)
    
    for split in splits:
        test_df = df.iloc[split['test_start']:split['test_end']]
        test_mask = mask.iloc[split['test_start']:split['test_end']]
        
        # Non-overlapping trade selection
        next_allowed = test_df.index.min()
        for idx in test_df.index:
            if idx < next_allowed or not bool(test_mask.loc[idx]):
                continue
            if target in test_df.columns:
                val = test_df.at[idx, target]
                if not pd.isna(val):
                    all_returns.append(float(val) - cost)
            next_allowed = idx + hold_bars
    
    if not all_returns:
        return {'trades': 0, 'mean_return': 0, 'total_return': 0, 'win_rate': 0}
    
    r = np.array(all_returns)
    return {
        'trades': len(r),
        'mean_return': float(r.mean()),
        'total_return': float(r.sum()),
        'win_rate': float((r > 0).mean()),
    }

# Run the sweep
results = []
combos = list(product(
    param_grid['liq_z_threshold'],
    param_grid['liq_imbalance_min'],
    param_grid['crowding_cap'],
    param_grid['oi_divergence_min'],
))

print(f'Running {len(combos)} parameter combinations...')
for i, (liq_z, liq_imb, crowd, oi_div) in enumerate(combos):
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{len(combos)}...')
    r = evaluate_strategy(df, splits, liq_z, liq_imb, crowd, oi_div)
    r['liq_z'] = liq_z
    r['liq_imb'] = liq_imb
    r['crowd_cap'] = crowd
    r['oi_div'] = oi_div
    results.append(r)

sweep_df = pd.DataFrame(results)
print(f'\nDone! {len(sweep_df)} combinations tested.')
print(f'Combinations with at least 1 trade: {(sweep_df["trades"] > 0).sum()}')

### Step 4: The "best" strategy

Now let's find the winner. Sort by total return and look at the top results.

**This is the moment where most researchers get fooled.**

In [ ]:
# Filter to strategies with at least 5 trades
viable = sweep_df[sweep_df['trades'] >= 5].copy()
viable = viable.sort_values('total_return', ascending=False)

print('='*70)
print('  TOP 10 PARAMETER COMBINATIONS BY TOTAL RETURN')
print('='*70)
print()

for i, (_, row) in enumerate(viable.head(10).iterrows(), 1):
    print(f'  #{i}  Total: {row["total_return"]*100:+.2f}%  '
          f'Trades: {row["trades"]:.0f}  '
          f'Win: {row["win_rate"]*100:.0f}%  '
          f'Avg: {row["mean_return"]*100:+.3f}%')
    print(f'       liq_z>{row["liq_z"]}  imb>{row["liq_imb"]}  '
          f'crowd<{row["crowd_cap"]}  oi_div>{row["oi_div"]}')
    print()

best = viable.iloc[0]
print('='*70)
print(f'  WINNER: {best["total_return"]*100:+.2f}% total return, '
      f'{best["win_rate"]*100:.0f}% win rate, {best["trades"]:.0f} trades')
print('='*70)
print()
print('Looks great, right?')
print()
print('Now go to Part 2 to see what happens when we test this')
print('on data the sweep has NEVER seen...')

---

### What just happened

You tested **400 parameter combinations** on the same walk-forward test folds.

Think about it:
- If each combination had a 50/50 chance of being positive on any given fold...
- With 400 tries, the probability that **at least one** looks amazing is nearly 100%
- The "best" combination isn't good because the strategy works
- It's "good" because with 400 lottery tickets, you're guaranteed a winner

This is called **selection bias** or **data snooping**.

**The walk-forward validation was honest.** The problem isn't the validation — it's that you picked the best out of 400 candidates AFTER seeing the results.

In the next notebook, we'll prove this by testing the "winner" on data it has never seen.

---

**Key takeaway:** A parameter sweep doesn't find good strategies. It finds strategies that got lucky on your specific data.

→ Continue to **Part 2: The Holdout Test**